# ResNet-50 5-Fold Baseline — Google Colab

This notebook runs the baseline without segmentation guidance using the same workflow as the guided notebook. Create the archive from the repository root so its internal paths match the local layout:

```bash
tar -czf dataset_gcolab.tar.gz \
  000_dataset/_segmentation_dataset_v2/004_classification_cv_5fold_seed42.csv \
  000_dataset/_segmentation_dataset_v2/ct_windowed \
  000_dataset/_segmentation_dataset_v2/mask
sha256sum dataset_gcolab.tar.gz > dataset_gcolab.tar.gz.sha256
```

## 1. GPU and Google Drive

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU through Runtime > Change runtime type.")
from google.colab import drive
drive.mount("/content/drive")
print(torch.cuda.get_device_name(0))

## 2. Runtime configuration

In [ ]:
from pathlib import Path
REPOSITORY_URL = "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/mask-guided-lung-nodule-xai")
DRIVE_DATASET_ARCHIVE = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz"
DRIVE_DATASET_CHECKSUM = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz.sha256"
LOCAL_DATA_ROOT = Path("/content/classification_data")
EXPERIMENT_ID = "dc730a13-5813-4d87-b15c-3b630deb32b5"
DRIVE_EXPERIMENT_ROOT = DRIVE_PROJECT_DIR / "experiment_results" / EXPERIMENT_ID
DRIVE_OUTPUT_DIR = DRIVE_EXPERIMENT_ROOT / "classification/cv_resnet50"
BATCH_SIZE = 32  # reduce if an OOM occurs
NUM_WORKERS = 0
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20
FORCE_REEXTRACT = False
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 3. Clone the source and install dependencies

In [ ]:
import subprocess, sys
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPOSITORY_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations==2.0.8", "zennit==0.5.1"], check=True)
if not (PROJECT_ROOT / "003_classification/cv_resnet50/train.py").is_file():
    raise FileNotFoundError("Push the baseline source to the branch first.")

## 4. Validate and extract the dataset

In [ ]:
import hashlib
if not DRIVE_DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(DRIVE_DATASET_ARCHIVE)
if DRIVE_DATASET_CHECKSUM.is_file():
    expected = DRIVE_DATASET_CHECKSUM.read_text().split()[0].lower()
    digest = hashlib.sha256()
    with DRIVE_DATASET_ARCHIVE.open("rb") as source:
        for chunk in iter(lambda: source.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise RuntimeError("Archive checksum mismatch.")
members = [
    "000_dataset/_segmentation_dataset_v2/004_classification_cv_5fold_seed42.csv",
    "000_dataset/_segmentation_dataset_v2/ct_windowed",
    "000_dataset/_segmentation_dataset_v2/mask",
]
marker = LOCAL_DATA_ROOT / ".cv_resnet50_extracted"
if FORCE_REEXTRACT or not marker.is_file():
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(DRIVE_DATASET_ARCHIVE), "-C", str(LOCAL_DATA_ROOT), *members], check=True)
    marker.touch()
print("Dataset ready.")

## 5. Link the local layout and Drive outputs

In [ ]:
import os, importlib
local_dataset = LOCAL_DATA_ROOT / "000_dataset"
drive_classification = DRIVE_EXPERIMENT_ROOT / "classification"
drive_classification.mkdir(parents=True, exist_ok=True)
project_experiment = PROJECT_ROOT / "experiment_results" / EXPERIMENT_ID
project_experiment.mkdir(parents=True, exist_ok=True)
links = {PROJECT_ROOT / "000_dataset": local_dataset, project_experiment / "classification": drive_classification}
for link, target in links.items():
    if link.is_symlink():
        if link.resolve() != target.resolve(): raise RuntimeError(f"Symlink target mismatch: {link}")
    elif link.exists():
        raise FileExistsError(f"Path exists and is not a symlink: {link}")
    else:
        os.symlink(target, link, target_is_directory=True)
    print(f"{link} -> {target}")
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

## 6. Dataset and model preflight

In [ ]:
import pandas as pd
dataset_root = local_dataset / "_segmentation_dataset_v2"
metadata_path = dataset_root / "004_classification_cv_5fold_seed42.csv"
metadata = pd.read_csv(metadata_path)
missing_ct = [p for p in metadata["ct_windowed_path"] if not (dataset_root / str(p)).is_file()]
missing_mask = [p for p in metadata["mask_path"] if not (dataset_root / str(p)).is_file()]
if missing_ct or missing_mask:
    raise FileNotFoundError(f"Missing CT={len(missing_ct)}, mask={len(missing_mask)}")
dataset_module = importlib.import_module("003_classification.utils.dataset")
train_base = importlib.import_module("003_classification.fulltuning_resnet50.train")
sample_dataset = dataset_module.LungClassificationDataset(dataset_root, "val", train_base.build_val_transform(), metadata_path=metadata_path, cv_fold=0)
sample, target = sample_dataset[0]
model = train_base.build_model(2).eval()
with torch.no_grad(): output = model(sample.unsqueeze(0).to(train_base.DEVICE))
del model; torch.cuda.empty_cache()
print(len(metadata), tuple(sample.shape), tuple(output.shape), int(target))

## 7. Run five-fold training

In [ ]:
import json
config_path = PROJECT_ROOT / "003_classification/configs/cv_resnet50.json"
config = json.loads(config_path.read_text())
config["experiment"]["id"] = EXPERIMENT_ID
config["training"]["batch_size"] = BATCH_SIZE
config["training"]["num_epochs"] = NUM_EPOCHS
config["training"]["device"] = "cuda"
config["dataloader"]["num_workers"] = NUM_WORKERS
config["dataloader"]["persistent_workers"] = NUM_WORKERS > 0
config["early_stopping"]["patience"] = EARLY_STOPPING_PATIENCE
config_path.write_text(json.dumps(config, indent=4) + "\n")
importlib.invalidate_caches()
name = "003_classification.cv_resnet50.train"
train_module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
if train_module.OUTPUT_DIR.resolve() != DRIVE_OUTPUT_DIR.resolve():
    raise RuntimeError(f"Unexpected output path: {train_module.OUTPUT_DIR}")
print(f"Output: {train_module.OUTPUT_DIR}")
train_module.main()

## 8. Inspect results and run testing + XAI

In [ ]:
from IPython.display import display
if not DRIVE_OUTPUT_DIR.is_dir(): raise FileNotFoundError(DRIVE_OUTPUT_DIR)
summary = DRIVE_OUTPUT_DIR / "cv_summary.csv"
if summary.is_file(): display(pd.read_csv(summary))
MAX_TEST_SAMPLES = None  # use 8 for a smoke test
command = [sys.executable, "-m", "003_classification.cv_resnet50.test", str(DRIVE_OUTPUT_DIR), "--batch-size", "2", "--num-workers", "0", "--device", "cuda", "--dpi", "120"]
if MAX_TEST_SAMPLES is not None: command += ["--max-samples", str(MAX_TEST_SAMPLES)]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print(f"Test output: {DRIVE_OUTPUT_DIR / 'test'}")